# Phase 3 — Baseline Models for Temporal Link Prediction

**Objectif**: Etablir des points de comparaison avant d'implémenter TGN.

Chaque baseline capture un aspect différent des données:

| Baseline | Ce qu'il capture | Ce qu'il ignore |
|----------|-----------------|----------------|
| **Random** | Rien (borne inférieure) | Tout |
| **Historical Frequency** | Popularité statique des routes | Dynamique temporelle |
| **Recency** | Activité récente d'une route | Popularité globale |
| **EdgeBank** | Existence passée d'une route | Fréquence et temporalité |

---
## 1. Protocole d'évaluation: One-vs-Many MRR

### Comment fonctionne l'évaluation TGB?

Pour chaque interaction réelle (src, dst, t) dans le set de validation:
1. On a la vraie destination `dst` (positive)
2. TGB fournit 20 fausses destinations (negatives) pré-générées
3. Le modèle score les 21 candidats (1 positif + 20 négatifs)
4. On calcule le **rang** de la vraie destination parmi les 21
5. **MRR** = Moyenne de 1/rang sur tous les exemples

### Interprétation du MRR

- MRR = 1.0: la vraie destination est TOUJOURS classée #1 (parfait)
- MRR = 0.5: la vraie destination est en moyenne classée #2
- MRR = H(21)/21 ≈ 0.169: score aléatoire (chance level)

### Pourquoi pas Hits@10?

Avec seulement 21 candidats, Hits@10 est très facile à satisfaire (il suffit d'être dans le top 10 sur 21). MRR est la métrique officielle TGB car elle distingue mieux les modèles.

---
## 2. Description des Baselines

### Random
Score = nombre aléatoire. Sert de borne inférieure absolue. MRR théorique = H(21)/21 ≈ 0.169.

### Historical Frequency (HistFreq)
Score(src, dst) = nombre de fois que le vol (src→dst) est apparu dans les données d'entraînement.

**Intuition**: les routes fréquentes (ex: KJFK→EGLL) sont plus probables que les routes rares. C'est un prior de popularité statique.

### Recency
Score(src, dst, t) = 1 / (t - t_last) où t_last est le dernier timestamp observé pour (src, dst).

**Intuition**: une route avec un vol récent est plus susceptible d'en avoir un autre bientôt. Capture la dynamique temporelle locale.

### EdgeBank (unlimited)
Score(src, dst) = 1 si (src, dst) a déjà été observé, sinon 0.

**Intuition**: les routes qui ont existé dans le passé existeront encore. C'est le baseline standard de TGB (Poursafaei et al., NeurIPS 2022). La variante "unlimited" mémorise toutes les arêtes sans limite de fenêtre temporelle.

---
## 3. Résultats

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')

# Load results
with open('../results/baseline_results.json') as f:
    results = json.load(f)

print("Baseline Results (Validation Set)")
print("=" * 45)
print(f"{'Baseline':<22} {'MRR':>8} {'Hits@10':>8}")
print("-" * 42)
for name, res in results.items():
    print(f"{name:<22} {res['mrr']:>8.4f} {res['hits@10']:>8.4f}")

In [ ]:
# Bar chart comparison
names = list(results.keys())
mrrs = [results[n]['mrr'] for n in names]
hits = [results[n]['hits@10'] for n in names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = ['#9E9E9E', '#2196F3', '#FF9800', '#4CAF50']

# MRR chart
bars1 = ax1.bar(names, mrrs, color=colors, edgecolor='white', linewidth=1.5)
ax1.axhline(y=0.169, color='red', linestyle='--', alpha=0.5, label='Random expected')
ax1.set_ylabel('MRR', fontsize=12)
ax1.set_title('Mean Reciprocal Rank (higher = better)', fontsize=13)
ax1.set_ylim(0, 1)
for bar, v in zip(bars1, mrrs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')
ax1.legend(fontsize=10)

# Hits@10 chart
bars2 = ax2.bar(names, hits, color=colors, edgecolor='white', linewidth=1.5)
ax2.set_ylabel('Hits@10', fontsize=12)
ax2.set_title('Hits@10 (higher = better)', fontsize=13)
ax2.set_ylim(0, 1)
for bar, v in zip(bars2, hits):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
os.makedirs('../results/figures', exist_ok=True)
plt.savefig('../results/figures/10_baseline_comparison.png', dpi=150)
plt.show()
print('Saved: results/figures/10_baseline_comparison.png')

---
## 4. Analyse des résultats

### Observations clés

1. **HistFreq domine** (MRR 0.744): le réseau aérien est extrêmement régulier — les routes les plus fréquentes dans le passé restent les plus fréquentes dans le futur. C'est un baseline très difficile à battre.

2. **Recency est fort mais inférieur** (MRR 0.682): connaître le dernier vol sur une route est informatif, mais moins que la fréquence totale. Cela suggère que la *popularité structurelle* du réseau est plus prédictive que la *dynamique récente*.

3. **EdgeBank a un MRR faible** (0.160) malgré un Hits@10 élevé (0.928): c'est un score binaire (0 ou 1). Quand la vraie route ET la plupart des routes négatives existent toutes dans la mémoire, EdgeBank ne peut pas les départager — elles ont toutes score 1. Le rang est alors aléatoire parmi les candidats ayant le même score.

4. **Hits@10 est saturé**: trois baselines dépassent 0.92, confirmant que cette métrique ne discrimine pas bien avec 21 candidats. MRR est la métrique à suivre.

### Implication pour TGN

TGN doit battre MRR 0.744 (HistFreq) pour prouver sa valeur. Il doit combiner:
- La popularité des routes (capturée par HistFreq)
- La dynamique temporelle (partiellement capturée par Recency)
- Les patterns structurels du graphe (non capturés par les baselines)
- Les disruptions (COVID, changements saisonniers) que les baselines statiques ne gèrent pas

---
## 5. Modules produits

- `src/baselines/baselines.py`: Implémentations des 4 baselines + CommonNeighbors
- `src/baselines/evaluate.py`: Harness d'évaluation MRR avec TGB Evaluator
- `scripts/run_baselines.py`: Script d'évaluation complète (optimisé mémoire)
- `results/baseline_results.json`: Résultats sauvegardés

### Prochaine étape: Phase 4 — TGN

Le modèle TGN (Temporal Graph Network) doit exploiter la structure temporelle du graphe pour dépasser le MRR de 0.744 établi par HistFreq.

In [ ]:
print("Phase 3 - Baselines: COMPLETE")
print("=" * 45)
print(f"Target for TGN (Phase 4): MRR > 0.744")
print(f"\nBaseline to beat: Historical Frequency")
print(f"  MRR:     0.7438")
print(f"  Hits@10: 0.9273")